# Modül 1 — Regresyon

**Eğitmen: Elif Özkan**

[LinkedIn](https://www.linkedin.com/in/eliffozkan/) · [GitHub](https://github.com/eliffozkan) · [Eğitim reposu](https://github.com/eliffozkan/GDGC-AI_ML_Course-26_27)

[YouTube — Fütürist AI (kaynak)](https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence)

[Google Colab’da aç](https://colab.research.google.com/github/eliffozkan/GDGC-AI_ML_Course-26_27/blob/main/Module_1_Regression/Module_1_Regression_Code.ipynb)

**Ders 1 (80 dk):** girdi/çıktı ve tahmin 15, MSE/RMSE/MAE 15, vectorized GD 30, öğrenme oranı ve sklearn karşılığı 20 dk.

**Ders 2 (80 dk):** sigmoid/BCE 20, lojistik GD 25, karar sınırı 15, sklearn ve ödev bağlantısı 20 dk.

**Çalıştırma:** Temiz oturumda hücreleri üstten alta çalıştırın. Kod çıktılarını ilgili açıklamayla birlikte yorumlayın.

## Ders 1 — Sürekli değer tahmini

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
np.random.seed(42)

# MSE, RMSE ve MAE
Aynı tahminleri farklı kayıplarla ölçeriz. MSE büyük hataları kareyle büyütür; RMSE ve MAE hedefle aynı birimdedir.

In [ ]:
y = np.array([50,60,70,40,20,15,75,85,12,89],dtype=np.float32)
y_predicted = np.array([41.2,65,52.8,40.9,20,12.6,70.8,72.9,5.6,82.5],dtype=np.float32)
print(y.size,y_predicted.size)

In [ ]:
from sklearn.metrics import mean_squared_error

error = mean_squared_error(y_true=y,
                           y_pred=y_predicted)

print(error)

In [ ]:
def MSE_vector(y_true,y_pred):
    n = y_true.size
    error = y_true-y_pred
    error = error ** 2
    error = error.sum()
    error = error / n
    #or with just one line: return np.sum(np.square(y_pred-y_true)) / y_true.size
    return np.array(error,dtype=np.float64)

In [ ]:
error = MSE_vector(y_true=y, y_pred=y_predicted)
print("MSE:", error, "sklearn:", mean_squared_error(y, y_predicted))
print("RMSE:", np.sqrt(error), "MAE:", mean_absolute_error(y, y_predicted))

# Linear Regression Prediction

In [ ]:
w = 5
b = 10
inputs = np.array([5,10,15,20],dtype=np.float32)

In [ ]:
def predict(inputs,w=0,b=0):
    output = w * inputs + b
    return output
predict(inputs,w,b)

In [ ]:
x =  np.array([[1,2,3],[4,5,6],[-1,-2,-3],[-4,-5,-6]])
w = np.array([[1],[2],[3]])
print(x.shape,w.shape)
print(np.matmul(x,w))

# Linear Regression - Gradient Descent

In [ ]:
x = np.random.random((200, 2)) * 10
weights = np.array([[2.0], [3.0]])  # Yalnızca sentetik veri üretiminde biliniyor.
y = x @ weights + 10.0

## Vectorized Gradient Descent
Satırlar örnek, sütunlar özellik: `x` (n,m), `w` (m,1), `y` (n,1). Her adımda tüm örneklerin hatası tek matris işlemiyle hesaplanır. `for` yalnızca güncelleme adımlarını tekrarlar. `w` ve `b`, MSE gradyanının ters yönünde ilerler.

In [ ]:
def linear_regression_fit(input, output, alpha=0.01, iter=500):
    n, m = input.shape
    w = np.zeros((m, 1))
    b = 0.0
    history = []
    for _ in range(iter):
        error = input @ w + b - output
        history.append(float(np.mean(error ** 2)))
        w -= alpha * (2 * input.T @ error / n)
        b -= alpha * float(2 * error.mean())
    return w, b, np.array(history)

In [ ]:
w, b, history = linear_regression_fit(x, y, alpha=0.01, iter=1500)
print("Öğrenilen w:", w.ravel(), "b:", round(b, 3), "son MSE:", history[-1])
for alpha in [0.0001, 0.01, 0.02]:
    _, _, losses = linear_regression_fit(x, y, alpha=alpha, iter=100)
    plt.semilogy(losses, label=f"alpha={alpha}")
plt.xlabel("Güncelleme adımı"); plt.ylabel("MSE (log ölçek)")
plt.legend(); plt.show()

Loss azalırken sentetik veriyi üreten ağırlık ve bias’a yaklaşırız. Çok küçük öğrenme oranı yavaş ilerler; çok büyük oran loss’u büyütebilir. Normal equation başka bir çözüm yoludur; burada kodunu yazmıyoruz. Huber ve Smooth L1 büyük hataların etkisini sınırlar; bu dönem yalnızca mantığını bilmek yeterlidir.

In [ ]:
model_linear = LinearRegression().fit(x, y.ravel())
print("Sklearn w:", model_linear.coef_, "b:", model_linear.intercept_)
print("İlk iki tahmin:", model_linear.predict(x[:2]))

---
## Ders 2 — Sınıf olasılığı ve karar sınırı
Lineer çıktı şimdi sigmoid ile 0–1 aralığına çevrilir; MSE yerine BCE kullanılır.

# Binary Cross Entropy

In [ ]:
y_true = np.array([1,1,0])
y_pred = np.array([0.9,0.7,0.3])

In [ ]:
def bce(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-12, 1 - 1e-12)
    return float(-np.mean(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred)))

In [ ]:
bce(y_true,y_pred)

Bu örnekte doğru sınıfa verilen olasılık yükseldikçe BCE düşer. `clip` yalnızca `log(0)` sayısal sorununu önler; etiketler yine 0/1 kalır.

# Sigmoid

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# Logistic Regression Prediction

In [ ]:
def log_reg_predict(w,b,input):
    z = w*input + b
    y = sigmoid(z)
    return y

In [ ]:
log_reg_predict(2,1,0.5)

Çıktı sınıf etiketi değildir. `p >= 0.5` ise 1, aksi halde 0 seçeriz. Karar sınırı `X @ w + b = 0` doğrusudur.

# Logistic Regression - Gradient Descent

## Grafik fonksiyonu - Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_logistic_regression_line(X, y, w, b):
    """
    X: (n, 2) numpy array (2 feature)
    y: (n, 1) veya (n,) numpy array
    w: (2, 1) weight vector
    b: scalar bias
    """

    # y düzleştirme
    if y.ndim > 1:
        y = y.flatten()

    # Verileri çiz
    plt.figure(figsize=(8, 6))
    plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='red', label='Sınıf 0')
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='blue', label='Sınıf 1')

    # Karar sınırı çizgisi: w1*x1 + w2*x2 + b = 0  =>  x2 = -(w1*x1 + b)/w2
    x1_vals = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 100)
    w1, w2 = w[0, 0], w[1, 0]
    if w2 != 0:
        x2_vals = -(w1 * x1_vals + b) / w2
        plt.plot(x1_vals, x2_vals, 'k--', label='Karar Sınırı')
    else:
        plt.axvline(x=-b / w1, color='k', linestyle='--', label='Karar Sınırı (dikey)')

    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title("Logistic Regression Sınıflandırması")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
true_weights = np.array([[1.2], [-0.8]])
x = np.random.randn(200, 2)
y_prob = sigmoid(x @ true_weights + 0.4)
y = (y_prob > 0.5).astype(int)

## Vectorized lojistik eğitim
Sigmoid skoru bir olasılık üretir. BCE gradyanı `X.T @ (pred-y) / n` olur; bias da güncellenir. Eğitim sonunda 0.5 eşiğiyle sınıf seçilir.

In [ ]:
def logistic_regression_model_matrix(x, y, alpha=0.1, iter=1000):
    n, m = x.shape
    w = np.zeros((m, 1)); b = 0.0; history = []
    for _ in range(iter):
        pred = sigmoid(x @ w + b)
        history.append(bce(y, pred))
        error = pred - y
        w -= alpha * (x.T @ error / n)
        b -= alpha * float(error.mean())
    return w, b, np.array(history)

In [ ]:
w_matrix, b_matrix, history = logistic_regression_model_matrix(x, y)
y_predicted = (sigmoid(x @ w_matrix + b_matrix) >= 0.5).astype(int)
print("Eğitim doğruluğu:", np.mean(y_predicted == y))
plt.plot(history); plt.xlabel("Güncelleme adımı"); plt.ylabel("BCE"); plt.show()

## Results

In [ ]:
plot_logistic_regression_line(x, y, w_matrix, b_matrix)

Grafikte renkler gerçek sınıfı, kesikli çizgi 0.5 karar sınırını gösterir. Loss düşmesi eğitim verisine uyumu anlatır; yeni veride başarı ayrı değerlendirilir. Aynı sınıflandırmayı sklearn ile yapalım.

In [ ]:
model_logistic = LogisticRegression(max_iter=1000).fit(x, y.ravel())
print("İlk beş sınıf:", model_logistic.predict(x[:5]))
print("İlk beş sınıf-1 olasılığı:", model_logistic.predict_proba(x[:5])[:, 1])
plot_logistic_regression_line(x, y, model_logistic.coef_.reshape(2, 1), model_logistic.intercept_[0])

**Ödev bağlantısı:** `fit(X_train,y_train)` eğitim, `predict(X_test)` yeni örnek tahminidir. Ev fiyatı görevinde MSE; müşteri terk görevinde iki müşterinin sınıfı istenir. Modül 2’de bu sınıflandırmanın başarısını değerlendireceğiz.

**Modül materyalleri:** [Module_1_Regression](https://github.com/eliffozkan/GDGC-AI_ML_Course-26_27/tree/main/Module_1_Regression)

Ders kodu ve ödev dosyaları bu modül klasöründedir.